# 06 · Can we search photos with a sentence (and the other way around)?

Images and captions live in the same vector space, so CLIP can search both ways: type a caption and get photos back, or show it a photo and get captions back. We'll build a tiny catalog of eight COCO photos and try both directions.

**Read more in [the book](https://arxiv.org/abs/2605.07544):** Chapter 5, Sections 5.2.3, *Image-Text Retrieval*, and 5.3.3, *Retrieval Benchmarks and Metrics*.

> **How to use this notebook:** Run the cells top to bottom. When you hit a **Pause and predict**, take a guess before running the next cell. You'll remember way more that way. Cells marked **TODO** are yours to play with: change a value, re-run, and see what happens.

### Pick your GPU

Got a GPU (or a few)? Set `GPU_ID` in the next cell to the one you want. Run `nvidia-smi` in a terminal to see the list. Only have one? Leave it at `"0"`. No GPU at all? Set it to `""` and everything runs on the CPU. It's slower, but it works.

This cell has to run *before* PyTorch is imported, so if you change it later, restart the kernel and run from the top.

In [ ]:
# TODO: choose your GPU. This has to run before torch is imported.
#   "0"  -> the first GPU (leave this if you only have one)
#   "1"  -> the second GPU, and so on (`nvidia-smi` lists them)
#   ""   -> no GPU, run everything on the CPU
#   None -> don't restrict anything; PyTorch sees every GPU
GPU_ID = "0"

import os
if GPU_ID is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # number GPUs the same way nvidia-smi does
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 1. What do we need to get started?

### 1.1 Grab our tools

Same lineup as before, plus `textwrap.fill` to wrap long captions so they fit above small images. Everything about "what matches what" comes from CLIP's pretrained encoders. No keyword matching, no hand-built image features.

In [ ]:
from textwrap import fill  # wraps long captions so they fit above small images

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor

### 1.2 Pin down the catalog and the model

`DATASET_REVISION` pins an exact version of the dataset, so we get the same rows every time. Two more settings:

- `CATALOG_SIZE = 8`: how many photos (and captions) are in our catalog
- `TOP_K = 3`: how many results each search returns, best first

In [ ]:
DATASET_ID = "Erland/coco_captions_small"
DATASET_REVISION = "a516b5101cec691eb65f455ffdb054afb2fdd5dd"  # pin the exact dataset version

MODEL_ID = "openai/clip-vit-base-patch32"
CATALOG_SIZE, TOP_K = 8, 3  # catalog size, and how many results each search returns

One more bit of setup: where should the model and both kinds of queries live? GPU if we have one, CPU otherwise.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU if we have one, CPU otherwise
print("device:", device)  # "cuda" here means the GPU we picked at the top

## 2. How do we build a catalog where we know the right answers?

To grade a search engine, we need to know what the *right* result is for each query. So we want a catalog where every photo has exactly one caption that belongs to it.

### 2.1 Load real image–caption pairs

The photos come from **COCO** (short for *Common Objects in Context*), a big dataset of everyday scenes where people wrote several captions for each photo. That means several rows here can share one photo, each with a different caption. We'll fix that in a second.

Here's what COCO looks like, with one of the human-written captions under each photo:

<img src="../assets/diagrams/notebook6/book_fig5_9.png" alt="Example image-caption pairs from MS COCO Captions" width="520">

*Figure 5.9 from [the book](https://arxiv.org/abs/2605.07544): example pairs from MS COCO Captions.*

In [ ]:
dataset = load_dataset(
    DATASET_ID, split="train", revision=DATASET_REVISION
)
print(dataset)
print("columns:", dataset.column_names)  # we use image, caption, and cocoid

### 2.2 Keep the first caption for each photo

Every photo has an ID called `cocoid`. We walk through all the rows, and `setdefault` stores a value only the *first* time it sees a key. Later rows with the same `cocoid` get ignored. So we end up with exactly one row and one caption per distinct photo.

In [ ]:
first_row = {}  # cocoid -> index of its first row
first_caption = {}  # cocoid -> its first caption

for index, (cocoid, caption) in enumerate(zip(dataset["cocoid"], dataset["caption"])):
    first_row.setdefault(cocoid, index)  # setdefault only stores the first one it sees
    first_caption.setdefault(cocoid, caption.strip())
print("distinct photographs:", len(first_row))

### 2.3 Pick eight matching pairs

We take the first eight photos and collect their images and captions in the same order. That gives us one really handy rule: **image `i` and caption `i` belong together.** That's our answer key for grading later.

In [ ]:
catalog_ids = list(first_row)[:CATALOG_SIZE]  # the first CATALOG_SIZE photos, so the pick is the same every run
records = [dataset[first_row[cocoid]] for cocoid in catalog_ids]
images = [record["image"].convert("RGB") for record in records]
captions = [first_caption[cocoid] for cocoid in catalog_ids]  # same order as images: caption i goes with image i
print("catalog identifiers:", catalog_ids)

Let's look at the whole catalog before CLIP searches it. Each caption sits right above its own photo.

> **Pause and predict:** Some captions mention something really distinctive, while others describe pretty ordinary scenes. Which caption do you think will be easiest for CLIP to match to its photo? Which one might get mixed up with another photo?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(11, 6))  # each caption sits above its own photo
for axis, image, caption in zip(axes.flat, images, captions):
    axis.imshow(image)
    axis.set_title(fill(caption, 25), fontsize=8)
    axis.axis("off")
fig.suptitle("The eight-record retrieval catalog")
fig.tight_layout()
plt.show()

Hang on to your guesses. We'll check them against the similarity matrix in Section 4.

## 3. How do we turn the catalog into vectors?

### 3.1 Load CLIP

Same pretrained CLIP as Notebook 05, in eval mode. We're not training anything on these eight pairs. We're just using what CLIP already learned during pretraining.

In [ ]:
processor = CLIPProcessor.from_pretrained(MODEL_ID)  # the prep recipe that belongs to this checkpoint
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

### 3.2 Prep the photos and captions separately

Why two separate calls instead of one? This mirrors how a real search system works. The catalog gets encoded once, ahead of time, and each new query gets encoded when it shows up. You'll see that in action in the TODO in Section 5.

In [ ]:
image_inputs = processor(images=images, return_tensors="pt").to(device)
text_inputs = processor(text=captions, padding=True, return_tensors="pt").to(device)  # captions stay in catalog order

### 3.3 Get one unit-length vector per item

Both encoders hand back vectors that are already projected into the shared space. Then we normalize each one to length 1. That way, a dot product measures only how much two vectors point the same way (cosine similarity), and a vector can't score high just because it happens to be long.

In [ ]:
with torch.inference_mode():  # no gradients needed, we're just reading features
    image_output = model.get_image_features(**image_inputs)
    text_output = model.get_text_features(**text_inputs)
image_vectors = F.normalize(image_output.pooler_output, dim=-1)  # length 1 per row
text_vectors = F.normalize(text_output.pooler_output, dim=-1)

Quick sanity check. We want eight rows on each side, the same width on both, and lengths super close to 1.

In [ ]:
print("image vectors:", tuple(image_vectors.shape))  # eight rows on each side
print("text vectors: ", tuple(text_vectors.shape))
print("first vector norms:", image_vectors[0].norm().item(), text_vectors[0].norm().item())  # should both be about 1

## 4. How can one matrix handle both search directions?

Multiplying `[8, width]` by `[width, 8]` gives an `[image, caption]` matrix with all 64 comparisons in it:

- **row `i`** compares image `i` with every caption
- **column `j`** compares caption `j` with every image
- **diagonal cell `(i, i)`** is a photo with its own caption

So we don't need two different systems for the two directions. It's the same numbers either way. The only thing that changes is whether we rank along a row or down a column.

In [ ]:
similarity = image_vectors @ text_vectors.T  # [8, width] @ [width, 8] -> [8, 8], rows are images, columns are captions
print("similarity matrix:", tuple(similarity.shape))
print("mean observed-pair score:", similarity.diag().mean().item())  # average over the 8 true pairs

Let's draw all 64 scores as a heatmap, where brighter means more similar.

> **Pause and predict:** If CLIP is doing its job, where should the brightest cells be? And look back at your guess from the gallery: which row or column might have a bright cell somewhere it shouldn't?

In [ ]:
fig, axis = plt.subplots(figsize=(5, 4.5))  # tick numbers are catalog positions
view = axis.imshow(similarity.detach().cpu(), cmap="magma")
axis.set(xlabel="caption position", ylabel="image position", title="CLIP cosine similarity")
axis.set_xticks(range(CATALOG_SIZE))
axis.set_yticks(range(CATALOG_SIZE))
fig.colorbar(view, ax=axis, label="cosine similarity")
fig.tight_layout()
plt.show()

Ideally the diagonal lights up, since those are the true pairs. A bright diagonal is a good sign, but here's the thing about search: there's no magic "good score" cutoff. What matters is **rank**. Is the right answer the brightest cell *in its row* (or *in its column*)? A pair can have a fairly low score and still come out on top if everything else in that row is even lower.

## 5. How do we search photos with a caption?

This is the same kind of search people run on huge collections. Here's an example from LAION-5B, a dataset of billions of image–text pairs: each text query (Q) brings back its nearest image in CLIP's embedding space, shown with that image's own caption (C).

<img src="../assets/diagrams/notebook6/book_fig5_2.png" alt="Text queries and their nearest images in CLIP embedding space, from LAION-5B" width="700">

*Figure 5.2 from [the book](https://arxiv.org/abs/2605.07544): examples from LAION-5B.*

Our catalog has eight photos instead of billions, but the search is the same: one caption vector against every image vector.

### 5.1 Rank one caption's column

Let's use caption 3 as our query. Its own photo also happens to be at position 3, but CLIP doesn't know that. All it gets is the caption vector and the eight image vectors. `topk` grabs the `TOP_K` highest scores in that column and where they sit.

**TODO:** After you've run this section once, change `text_query` (0 to 7) and re-run the next two cells.

In [ ]:
text_query = 3  # TODO: try other captions (0 to 7)
query_scores, image_positions = similarity[:, text_query].topk(TOP_K)  # rank one column: this caption vs. every image
print("query:", captions[text_query])
print("observed paired image position:", text_query)
print("retrieved positions:", image_positions.tolist())

### 5.2 Look at the top three photos

Each title shows the rank, the photo's catalog position, and its cosine score. The photo that actually goes with the caption is in green.

If some other photo beats it, don't just shrug and say "the score says so". Look at the picture. What in it matches the caption? Often the "wrong" answer is a pretty reasonable match.

In [ ]:
fig, axes = plt.subplots(1, TOP_K, figsize=(9, 3), squeeze=False)  # best match on the left
for rank, (axis, position, score) in enumerate(zip(axes[0], image_positions, query_scores), 1):
    item = int(position)
    color = "green" if item == text_query else "black"
    axis.imshow(images[item])
    axis.set_title(f"#{rank} image {item}" + chr(10) + f"score {score:.3f}", color=color)
    axis.axis("off")
fig.suptitle(fill(captions[text_query], 70), fontsize=10)
fig.tight_layout()
plt.show()

### 5.3 Write your own query

So far our queries came from the catalog itself. But a real search engine gets brand-new queries all the time. Remember how we encoded the catalog separately in Section 3.2? This is why. A new caption only needs **one** trip through the text encoder. The eight image vectors are already done, so we just reuse them.

**TODO:** Write your own caption in `my_query` and re-run. Describe one of the eight photos in your own words, or try something that isn't in the catalog at all and see what CLIP comes up with.

In [ ]:
# TODO: write your own caption and re-run
my_query = "a person riding a bike down the street"

my_query_inputs = processor(text=[my_query], padding=True, return_tensors="pt").to(device)
with torch.inference_mode():
    my_query_vector = F.normalize(model.get_text_features(**my_query_inputs).pooler_output, dim=-1)  # [1, width]
my_scores, my_positions = (image_vectors @ my_query_vector.T).squeeze(1).topk(TOP_K)  # [8, 1] -> [8] -> top K

fig, axes = plt.subplots(1, TOP_K, figsize=(9, 3), squeeze=False)  # the catalog images are reused, not re-encoded
for rank, (axis, position, score) in enumerate(zip(axes[0], my_positions, my_scores), 1):
    axis.imshow(images[int(position)])
    axis.set_title(f"#{rank} image {int(position)}" + chr(10) + f"score {score:.3f}")
    axis.axis("off")
fig.suptitle(fill(my_query, 70), fontsize=10)
fig.tight_layout()
plt.show()

Notice that CLIP *always* returns `TOP_K` photos, even when nothing in the catalog really fits. Ranking always produces a winner. If you wanted a search engine that can say "no good match", you'd need something extra on top, like a score threshold.

<details>
<summary>Hint: queries worth trying</summary>

- Copy one of the catalog captions and change one detail (a color, a count, an object). Does the same photo still win?
- Describe a photo using words that *aren't* in its caption.
- Write something really vague, like `"a picture"`. How close are the top scores to each other?

</details>

## 6. How do we search captions with a photo?

### 6.1 Rank one image's row

Reverse search is the exact same idea, just sideways. Instead of column 3, we rank **row 3**: photo 3 against every caption. Then we use the positions we get back to look up the caption text.

> **Pause and predict:** Section 5 searched with caption 3. Now we search with photo 3. Will the results be a mirror image of each other? Say caption 3 found photo 3 at rank #1. Does that mean photo 3 has to find caption 3 at rank #1?

**TODO:** Change `image_query` (0 to 7) to search with a different photo.

In [ ]:
image_query = 3  # TODO: try other photos (0 to 7)
caption_scores, caption_positions = similarity[image_query].topk(TOP_K)  # rank one row: this photo vs. every caption
print("image query position:", image_query)
for rank, (position, score) in enumerate(zip(caption_positions, caption_scores), 1):
    item = int(position)
    print(f"{rank}. caption {item} | {score:.3f} | {captions[item]}")

Not necessarily. The two lists don't have to mirror each other. A cell can be the biggest number in its **row** without being the biggest in its **column**, because each query is up against a different set of competitors. Photo 3 competes against the other photos when caption 3 is the query, and caption 3 competes against the other captions when photo 3 is the query.

## 7. How good is the search overall?

One query at a time is nice for building intuition. But how do we score the whole system? We run all eight queries in both directions and count how often the right answer shows up near the top.

### 7.1 Rank every row and every column

`argsort(descending=True)` sorts each row and gives back the *positions* in order, best first.

- **Image → text:** each image row ranks the captions.
- **Text → image:** we transpose the matrix (`.T`) so each caption becomes a row, then rank the images the same way.

The answer key is easy: for query `i`, the right answer is position `i`, in both directions.

In [ ]:
image_to_text_order = similarity.argsort(dim=1, descending=True)  # each image row: caption positions, best first
text_to_image_order = similarity.T.argsort(dim=1, descending=True)  # each caption row: image positions, best first
targets = torch.arange(CATALOG_SIZE, device=device)  # query i's right answer is position i

### 7.2 Compute Recall@1 and Recall@3

**Recall@K** asks a simple question for each query: *did the right answer show up anywhere in the top K results?* Then it reports what fraction of queries got a yes. You met Recall@1 back in Notebook 03. Recall@3 is just more forgiving: the right answer can be at rank 1, 2, or 3.

This is also how the [CLIP paper](https://arxiv.org/abs/2103.00020) scores retrieval. Its Table 13 reports Recall@1, @5 and @10 in both directions on Flickr30k and MSCOCO. Same idea, just with thousands of test images instead of our eight.

> **Pause and predict:** Can Recall@3 ever come out *lower* than Recall@1?

In [ ]:
for k in (1, 3):
    image_hits = image_to_text_order[:, :k].eq(targets[:, None]).any(dim=1)  # a hit anywhere in the first k counts
    text_hits = text_to_image_order[:, :k].eq(targets[:, None]).any(dim=1)
    print(f"R@{k} image→text: {image_hits.float().mean():.1%}")
    print(f"R@{k} text→image: {text_hits.float().mean():.1%}")

No, it can't. The top three always include the top one, so every query that's a hit at K = 1 is automatically a hit at K = 3. Recall can only stay the same or go up as K grows.

With eight queries per direction, each query is worth 12.5%, so these numbers jump in big steps. Compare the two directions, too. Do they agree? If they don't, look back at Section 6 for why that's allowed.

> **Something to chew on:** Every COCO photo has several valid captions, but we kept only one per photo. So if photo 3's *other* caption were in the catalog and ranked first for photo 3, Recall@1 would count that as a miss, even though it's a perfectly good answer. How would you fix the answer key for a catalog with several captions per photo?

## Recap

That's the whole trip. One catalog, one matrix, two search directions:

```text
8 photos     → CLIP image encoder + normalize → image_vectors  [8, 512]
8 captions   → CLIP text encoder + normalize  → text_vectors   [8, 512]
                          ↓
          image_vectors @ text_vectors.T
                          ↓
          similarity  [8 images, 8 captions]
     rank a column  →  caption query  →  top photos
     rank a row     →  photo query    →  top captions
     where does the diagonal partner land?  →  Recall@1, Recall@3
```

**The big ideas:**

- A search system can **encode its catalog once** and then answer new queries with a single encoding plus a vector comparison.
- **One `[image, caption]` similarity matrix** handles both image → text and text → image search. Rows for one, columns for the other.
- **Search is about rank, not raw scores.** A score only means something compared with the other candidates for the same query.
- **Recall@K** asks whether the right answer lands in the top K. Bigger K can never lower it.
- Our eight-item catalog is built for looking at, not for benchmarking. A real evaluation would use a much bigger held-out set and count every valid caption per photo.

### Check your understanding

1. Why can a single similarity matrix serve both search directions?
2. Caption 3 retrieves photo 3 at rank #1, but photo 3 retrieves some other caption at rank #1. How is that possible?
3. In our eight-item catalog, what's Recall@8, and why is it a useless number?

<details>
<summary>Show answers</summary>

1. Cell `(i, j)` is the cosine similarity between image `i` and caption `j`, and that's the same number no matter which side the query is on. Image queries rank along a row, caption queries rank down a column.
2. Each query faces different competitors. Caption 3 only has to beat the other photos in column 3. Photo 3 has to beat the other captions in row 3, and one of them might score even higher with it.
3. It's always 100%. The top eight is the entire catalog, so the right answer is in there no matter what. A Recall@K tells you something only when K is a lot smaller than the catalog.

</details>

## Try it yourself

### TODO 1: Search with a photo that isn't in the catalog

In Section 5.3 you wrote a brand-new *caption*. Now let's try a brand-new *photo*. The cell below takes a photo from just past the end of our catalog, encodes it, and ranks our eight captions against it. Its real caption isn't in the catalog, so there's no "right" answer here. The question is whether the top captions still make sense.

In [ ]:
# TODO: change new_photo_offset to try different photos outside the catalog
new_photo_offset = 0
new_cocoid = list(first_row)[CATALOG_SIZE + new_photo_offset]
new_image = dataset[first_row[new_cocoid]]["image"].convert("RGB")

new_image_inputs = processor(images=[new_image], return_tensors="pt").to(device)
with torch.inference_mode():
    new_image_vector = F.normalize(model.get_image_features(**new_image_inputs).pooler_output, dim=-1)  # [1, width]
new_scores, new_positions = (new_image_vector @ text_vectors.T).squeeze(0).topk(TOP_K)  # [1, 8] -> [8] -> top K

fig, axis = plt.subplots(figsize=(3, 3))
axis.imshow(new_image)
axis.set_title(fill(first_caption[new_cocoid], 35), fontsize=8)  # its own caption, which isn't in the catalog
axis.axis("off")
plt.show()

for rank, (position, score) in enumerate(zip(new_positions, new_scores), 1):
    print(f"{rank}. caption {int(position)} | {score:.3f} | {captions[int(position)]}")

### TODO 2: Throw in some distractor captions

Right now each photo only competes against seven other captions. What if we pad the catalog with extra captions that don't belong to any photo? The cell below adds your **distractors** after the real captions (so the real ones keep positions 0 to 7) and recomputes image → text recall.

In [ ]:
# TODO: write captions that might steal the top spot from the real ones
my_distractors = ["a photo of something.", "a picture taken outdoors."]

distractor_inputs = processor(text=my_distractors, padding=True, return_tensors="pt").to(device)
with torch.inference_mode():
    distractor_vectors = F.normalize(model.get_text_features(**distractor_inputs).pooler_output, dim=-1)
bigger_text_vectors = torch.cat([text_vectors, distractor_vectors])  # real captions keep positions 0 to CATALOG_SIZE - 1
bigger_order = (image_vectors @ bigger_text_vectors.T).argsort(dim=1, descending=True)

for k in (1, 3):
    bigger_hits = bigger_order[:, :k].eq(targets[:, None]).any(dim=1)
    print(f"R@{k} image→text with {len(my_distractors)} distractors: {bigger_hits.float().mean():.1%}")

Compare these with the image → text numbers from Section 7.2. Adding candidates can only hurt or tie: a distractor can push a real caption down the list, but never up.

<details>
<summary>Hint</summary>

Vague distractors like the defaults usually lose to a specific caption. The sneaky ones are **near-duplicates**: copy a real caption from the gallery and change one detail, like a color or a number. Can CLIP still tell the real one from the fake?

</details>

### TODO 3: Grow the catalog

Go back to Section 1.2 and change `CATALOG_SIZE` from 8 to 16 or 32, then restart and Run All. It's still quick, since we're only encoding a few dozen items.

A few things to know:

- The gallery in Section 2.3 still shows only the first eight photos (it has eight slots), but everything else uses the full catalog.
- Check that "distinct photographs" in Section 2.2 is at least as big as your new `CATALOG_SIZE`.

Before you run it, guess: will Recall@1 go up or down? Why?

<details>
<summary>Hint</summary>

Every query now has more competitors, so there are more chances for some other item to sneak above the right answer. You'd usually expect recall to drop as the catalog grows. That's why reported Recall@K numbers always come with the catalog size. A Recall@1 on 8 items and one on 5,000 items are very different achievements.

</details>

---

**Up next, Notebook 07:** so far we've only *ranked* captions that already exist. What if we want a model to *write* a brand-new caption, one token at a time? We'll build a small captioner whose text decoder looks at the image while it writes, using **cross-attention**.